### Chapter 8.6 Promises

https://cs3110.github.io/textbook/chapters/ds/promises.html

In the functional programming paradigm, one of the best known abstractions for concurrency is promises.

A promise has three states (Pending, Resolved, and Rejected).

A promise has a resolver. The resolver for a promise will be used internally by the concurrency library but not revealed to clients. The clients will only get access to the promise.

There are two widely-used libraries in OCaml that implement promises: `Async` and `Lwt`. `Async` is developed by Jane Street. `Lwt` is part of the Ocsigen project, which is a web framework for OCaml.

Don’t think of `Lwt` as having anything to do with threads: it really is a library for promises.

In `Lwt`, a promise is a write-once reference: a value that is permitted to mutate at most once.

The promise abstraction by itself is not inherently concurrent. It’s just a data structure that can be written at most once, and that provides a means to control who can write to it (through the resolver).

The `Lwt` example is easier to understand even though the Cornell book Promise interface is supposed to be cleaner, because there is a concrete usage example of `Lwt` in the textbook while there is not example application of the Cornell book Promise module.

In [22]:
(* Making Our Own Lwt-style Promises *)
(** A signature for Lwt-style promises, with better names *) (* Lwt: Light Weight Threads *)
module type PROMISE = sig
  type 'a state =
    | Pending
    | Resolved of 'a
    | Rejected of exn

  type 'a promise

  type 'a resolver

  (** [make ()] is a new promise and resolver. The promise is pending. *)
  val make : unit -> 'a promise * 'a resolver

  (** [return x] is a new promise that is already resolved with value
      [x]. *)
  val return : 'a -> 'a promise

  (** [state p] is the state of the promise *)
  val state : 'a promise -> 'a state

  (** [resolve r x] resolves the promise [p] associated with [r] with
      value [x], meaning that [state p] will become [Resolved x].
      Requires: [p] is pending. *)
  val resolve : 'a resolver -> 'a -> unit

  (** [reject r x] rejects the promise [p] associated with [r] with
      exception [x], meaning that [state p] will become [Rejected x].
      Requires: [p] is pending. *)
  val reject : 'a resolver -> exn -> unit
end

module type PROMISE =
  sig
    type 'a state = Pending | Resolved of 'a | Rejected of exn
    type 'a promise
    type 'a resolver
    val make : unit -> 'a promise * 'a resolver
    val return : 'a -> 'a promise
    val state : 'a promise -> 'a state
    val resolve : 'a resolver -> 'a -> unit
    val reject : 'a resolver -> exn -> unit
  end


### Questions: What's the difference between promises (delayed/deferred computation) and lazy evaluation?


In [23]:
type 'a state = Pending | Resolved of 'a | Rejected of exn
type 'a promise = 'a state ref

type 'a state = Pending | Resolved of 'a | Rejected of exn


type 'a promise = 'a state ref


In [24]:
type 'a resolver = 'a promise

type 'a resolver = 'a promise


In [25]:
(** [write_once p s] changes the state of [p] to be [s].  If [p] and [s]
    are both pending, that has no effect.
    Raises: [Invalid_arg] if the state of [p] is not pending. *)
let write_once p s =
  if !p = Pending
  then p := s
  else invalid_arg "cannot write twice"

val write_once : 'a state ref -> 'a state -> unit = <fun>


#### Function make is internal. It creates a promise and a resolver. Note that the resolver is used by the concurrency library, not the client code.

In [26]:
let make () =
  let p = ref Pending in
  (p, p)


  (* Ch7.1 References *)

val make : unit -> 'a state ref * 'a state ref = <fun>


In [27]:
(* The complete implementation of our own Lwt-style Promise *)
module Promise : PROMISE = struct
  type 'a state =
    | Pending
    | Resolved of 'a
    | Rejected of exn  (* Exception *)

  type 'a promise = 'a state ref

  type 'a resolver = 'a promise

  (** [write_once p s] changes the state of [p] to be [s]. If [p] and
      [s] are both pending, that has no effect. Raises: [Invalid_arg] if
      the state of [p] is not pending. *)
  let write_once p s =
    if !p = Pending then p := s else invalid_arg "cannot write twice"

  let make () =
    let p = ref Pending in
    (p, p)

  let return x = ref (Resolved x)

  let state p = !p

  let resolve r x = write_once r (Resolved x)

  let reject r x = write_once r (Rejected x)
end

module Promise : PROMISE


### Lwt - Lightweight Threads - A library for promises, not system-level threads

* https://ocsigen.org/lwt/latest/manual/manual
* https://ocsigen.org/lwt/5.4.2/api/Lwt

In [28]:
#use "topfind"

- : unit = ()
Findlib has been successfully loaded. Additional directives:
  #require "package";;      to load a package
  #list;;                   to list the available packages
  #camlp4o;;                to load camlp4 (standard syntax)
  #camlp4r;;                to load camlp4 (revised syntax)
  #predicates "p,q,...";;   to set these predicates
  Topfind.reset();;         to force that packages will be reloaded
  #thread;;                 to enable threads

- : unit = ()


In [29]:
(* On codespaces, you may need to run this line instead if the line above is not successful. *)

#use "/home/codespace/.opam/default/lib/toplevel/topfind"  


- : unit = ()
Findlib has been successfully loaded. Additional directives:
  #require "package";;      to load a package
  #list;;                   to list the available packages
  #camlp4o;;                to load camlp4 (standard syntax)
  #camlp4r;;                to load camlp4 (revised syntax)
  #predicates "p,q,...";;   to set these predicates
  Topfind.reset();;         to force that packages will be reloaded
  #thread;;                 to enable threads

- : unit = ()


In [31]:
#require "lwt";;

In [32]:
#require "lwt.unix";;

In [33]:
open Lwt

In [34]:
open Lwt.Syntax

In [35]:
open Lwt_io

In [36]:
Lwt.wait()

- : '_weak2 Lwt.t * '_weak2 Lwt.u = (<abstr>, <abstr>)


In [37]:
(* a promise and a resolver -- wait is equivalent to make *)
let p, r = Lwt.wait();;

val p : '_weak3 Lwt.t = <abstr>
val r : '_weak3 Lwt.u = <abstr>


In [38]:
(* You can also specify that an int is expected. *)

let (p : int Lwt.t), r = Lwt.wait ()

val p : int Lwt.t = <abstr>
val r : int Lwt.u = <abstr>


In [39]:
(* Sleep is pending *)

Lwt.state p

- : int Lwt.state = Sleep


In [40]:
(* The resolver is now activated. *)

Lwt.wakeup r 42

- : unit = ()


In [41]:
Lwt.state p;;

- : int Lwt.state = Return 42


In [42]:
(* One cannot wake it up twice. Exception will be raised. *)
Lwt.wakeup r 45

error: runtime_error

In [43]:
(* An example of rejecting a promise. *)

let (p : int Lwt.t), r = Lwt.wait ();;
Lwt.wakeup_exn r (Failure "nope");; (* exn: exception *)
Lwt.state p;;

val p : int Lwt.t = <abstr>
val r : int Lwt.u = <abstr>


- : unit = ()


- : int Lwt.state = Fail (Failure "nope")


In [44]:
(* One cannot wakeup a promise that's already rejected. *)
Lwt.wakeup r 42

error: runtime_error

In [45]:
Lwt.state p;;

- : int Lwt.state = Fail (Failure "nope")


In [46]:
(* https://ocsigen.org/lwt/latest/api/Lwt#2_Cancellation

Promises created by Lwt.task can be resolved (specifically, rejected) by canceling them directly, in addition to being resolved through their paired resolvers.

In contrast, promises returned by Lwt.wait can only be resolved through their resolvers.

Overall, Lwt implements Promises fully, but is much more than just Promises. 
To study the abstraction functions relevant to Promises, focus on `Lwt.wait` and `Lwt.wakeup` and `Lwt.wakeup_exn`. DO NOT use Lwt.cancel.


*)

let (p : int Lwt.t), r = Lwt.wait ();;
Lwt.state p;;
Lwt.cancel p;;
Lwt.state p;;
Lwt.wakeup r 45;;
Lwt.state p;;
Lwt.cancel p;;
Lwt.state p;;

val p : int Lwt.t = <abstr>
val r : int Lwt.u = <abstr>


- : int Lwt.state = Sleep


- : unit = ()


- : int Lwt.state = Sleep


- : unit = ()


- : int Lwt.state = Return 45


- : unit = ()


- : int Lwt.state = Return 45


In [47]:
let (p : int Lwt.t), r = Lwt.wait ();;
Lwt.state p;;
Lwt.wakeup_exn r (Failure "some failure");;
Lwt.state p;;
Lwt.wakeup r 45;;
Lwt.state p;;

val p : int Lwt.t = <abstr>
val r : int Lwt.u = <abstr>


- : int Lwt.state = Sleep


- : unit = ()


- : int Lwt.state = Fail (Failure "some failure")


error: runtime_error

In [48]:
let print_the_int int = Lwt_io.printf "The int is: %d\n" int

val print_the_int : int -> unit Lwt.t = <fun>


In [49]:
(* The rest of the demo code won't work in Jupyter notebook because it is async. Use the promises.ml file instead. *)


Lwt.bind p print_the_int


error: runtime_error

In [50]:
Lwt.wakeup r 42

error: runtime_error

In [51]:
(* Define the delay function *)
let delay (sec : float) : unit Lwt.t =
  Lwt_unix.sleep sec

val delay : float -> unit Lwt.t = <fun>


In [52]:
let delay_then_print (sec: float) : unit Lwt.t =
  delay sec >>= fun () ->
  Lwt_io.printl "done"

val delay_then_print : float -> unit Lwt.t = <fun>


In [53]:
delay 3.

- : unit = ()


In [54]:
delay_then_print 3.0

done


- : unit = ()
